# Vodafone Customer Churn Prediction

End-to-end workflow: data audit, EDA, leakage-safe preprocessing, model comparison (Logistic Regression, Random Forest, XGBoost), evaluation and retention-oriented analysis.

All heavy lifting lives in the `churn` package (`src/churn`), so the notebook and the command line (`python -m churn.train`) run exactly the same logic.

> **Data.** Place the Kaggle file at `data/raw/customer_churn_data.xlsx` (see `data/README.md`). If it is missing, this notebook falls back to **synthetic demo data** so it still runs. Demo results are meaningless and are flagged loudly below.

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from churn import evaluate as ev
from churn.config import RANDOM_STATE
from churn.data import load_data
from churn.features import clean, column_groups, prepare_xy
from churn.models import build_models
from churn.synthetic import make_synthetic

warnings.filterwarnings("ignore", category=UserWarning)
sns.set_theme(style="whitegrid")

## 1. Load data

In [ ]:
DATA_PATH = ROOT / "data" / "raw" / "customer_churn_data.xlsx"
USE_DEMO_DATA = not DATA_PATH.exists()

if USE_DEMO_DATA:
    print("!" * 70)
    print("DATA FILE NOT FOUND -> using SYNTHETIC demo data. Results are NOT real.")
    print("!" * 70)
    raw = make_synthetic(2000, seed=RANDOM_STATE)
else:
    raw = load_data(DATA_PATH)

print(f"{raw.shape[0]:,} rows x {raw.shape[1]} columns")
raw.head()

## 2. Data quality audit

Which columns have gaps, and how many rows would the original notebook's `dropna()` have thrown away? The pipeline keeps every row and imputes inside the model pipeline instead.

In [ ]:
df = clean(raw)

missing = df.isna().sum()
audit = pd.DataFrame({"missing": missing, "pct": (missing / len(df) * 100).round(2)})
audit[audit["missing"] > 0].sort_values("missing", ascending=False)

In [ ]:
rows_with_gaps = df.isna().any(axis=1).sum()
print(f"Rows with at least one missing feature: {rows_with_gaps:,} of {len(df):,} ({rows_with_gaps / len(df):.1%})")
print(f"Duplicate customerIDs: {df['customerID'].duplicated().sum()}")
print(f"Churn rate (all rows): {(df['Churn'] == 'Yes').mean():.2%}")

## 3. Exploratory analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.countplot(data=df, x="Contract", hue="Churn", ax=axes[0], palette="Set2")
axes[0].set_title("Churn by contract type")

sns.boxplot(data=df, x="Churn", y="MonthlyCharges", hue="Churn", ax=axes[1], palette="Set2", legend=False)
axes[1].set_title("Monthly charges vs churn")

sns.histplot(data=df, x="tenure", hue="Churn", multiple="stack", bins=20, ax=axes[2], palette="Set2")
axes[2].set_title("Tenure distribution by churn")

plt.tight_layout()
plt.show()

In [ ]:
X, y = prepare_xy(raw)
eda = X.assign(Churn=y)

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5), sharey=True)
for ax, col in zip(axes, ["Contract", "InternetService", "PaymentMethod"]):
    rate = eda.groupby(col)["Churn"].mean().sort_values(ascending=False) * 100
    sns.barplot(x=rate.values, y=rate.index, color="#2F5597", ax=ax)
    ax.set_title(f"Churn rate (%) by {col}")
    ax.set_ylabel("")
    ax.axvline(y.mean() * 100, color="k", ls="--", lw=1)
plt.tight_layout()
plt.show()

## 4. Model comparison

Preprocessing (median imputation + missing-value flags + scaling for numerics, `Unknown` + one-hot for categoricals) is part of each pipeline, so it is fitted on training folds only. Models are ranked by cross-validated ROC-AUC on the **training** split; the test set is only used for the final report.

In [ ]:
numeric, categorical = column_groups(X)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
models = build_models(numeric, categorical, pos_weight=pos_weight)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rows, fitted = [], {}
for name, pipe in models.items():
    cv_auc = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc")
    pipe.fit(X_train, y_train)
    m = ev.evaluate_model(pipe, X_test, y_test)
    rows.append({"Model": name, "CV ROC-AUC": cv_auc.mean(), "Test ROC-AUC": m["roc_auc"],
                 "Test PR-AUC": m["pr_auc"], "Precision": m["precision"], "Recall": m["recall"],
                 "F1": m["f1"], "Top-decile lift": m["top_decile_lift"]})
    fitted[name] = pipe

results = pd.DataFrame(rows).set_index("Model").round(3)
best = results["CV ROC-AUC"].idxmax()
print(f"Best model by CV ROC-AUC: {best}")
results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name, pipe in fitted.items():
    RocCurveDisplay.from_estimator(pipe, X_test, y_test, name=name, ax=axes[0])
    PrecisionRecallDisplay.from_estimator(pipe, X_test, y_test, name=name, ax=axes[1])
axes[0].plot([0, 1], [0, 1], "k--")
axes[0].set_title("ROC curves (test set)")
axes[1].axhline(y_test.mean(), color="k", ls="--")
axes[1].set_title("Precision-recall curves (test set)")
plt.tight_layout()
plt.show()

## 5. What drives churn?

In [ ]:
tree_name = "XGBoost" if "XGBoost" in fitted else "Random Forest"
imp = ev.feature_importance(fitted[tree_name], top_n=12)

plt.figure(figsize=(9, 5))
sns.barplot(data=imp, x="Importance", y="Feature", color="#2F5597")
plt.title(f"Top 12 drivers of churn ({tree_name})")
plt.tight_layout()
plt.show()

In [ ]:
lr = fitted["Logistic Regression"]
names = lr.named_steps["prep"].get_feature_names_out()
coef = pd.Series(lr.named_steps["clf"].coef_.ravel(), index=names).sort_values()
pd.concat([coef.head(8).rename("coef (lowers churn risk)"), coef.tail(8).rename("coef (raises churn risk)")], axis=1)

Logistic-regression coefficients are on standardised features, so magnitudes are comparable across numeric columns. They describe association in this dataset, not causation.

## 6. Retention targeting: lift by risk decile

If a retention team can only contact a fraction of customers, how much better is the model than random selection?

In [ ]:
best_pipe = fitted[best]
scored = pd.DataFrame({"prob": best_pipe.predict_proba(X_test)[:, 1], "churned": y_test.values})
scored["decile"] = pd.qcut(scored["prob"].rank(method="first", ascending=False), 10, labels=range(1, 11))

table = scored.groupby("decile", observed=True).agg(customers=("churned", "size"), churn_rate=("churned", "mean"))
table["lift"] = table["churn_rate"] / scored["churned"].mean()
table["cum_churners_captured"] = table["churn_rate"].mul(table["customers"]).cumsum() / scored["churned"].sum()
table.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(x=table.index.astype(int), y=table["churn_rate"] * 100, color="#2F5597", ax=ax)
ax.axhline(scored["churned"].mean() * 100, color="k", ls="--", label="Overall churn rate")
ax.set(xlabel="Risk decile (1 = highest predicted risk)", ylabel="Actual churn rate (%)", title="Churn rate by predicted-risk decile")
ax.legend()
plt.tight_layout()
plt.show()

## 7. Next steps

- Calibrate probabilities and choose a decision threshold from the cost of a retention offer versus the value of a saved customer.
- Add SHAP explanations for customer-level reasons.
- Review sensitive features (`gender`, `SeniorCitizen`) before any real-world use.